In [2]:
# === Load Data ===
X_train = pd.read_csv(train_X_path)
y_df = pd.read_csv(train_y_path)
X_test = pd.read_csv(test_X_path)
test_ids = pd.read_excel(test_cat_path)['participant_id']

y_adhd = y_df['ADHD_Outcome']
y_sex  = y_df['Sex_F']

# === Data Cleaning ===
print("🧹 Starting data cleaning...")

# 1. Drop duplicate rows (if any)
X_train = X_train.drop_duplicates()
X_test = X_test.drop_duplicates()

# 2. Handle missing values
# - Fill numeric columns with median
for col in X_train.select_dtypes(include=[np.number]).columns:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# - Fill categorical columns with mode
for col in X_train.select_dtypes(exclude=[np.number]).columns:
    mode_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

# 3. Encode categorical features
from sklearn.preprocessing import LabelEncoder
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# 4. Remove constant/low-variance features
from sklearn.feature_selection import VarianceThreshold
var_thresh = VarianceThreshold(threshold=0.0)
X_train = pd.DataFrame(var_thresh.fit_transform(X_train), columns=X_train.columns[var_thresh.get_support()])
X_test = pd.DataFrame(var_thresh.transform(X_test), columns=X_test.columns[var_thresh.get_support()])

# 5. Scale numeric features (optional, since tree models don’t require it, but helps with stacking/logistic)
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X_train[X_train.columns] = scaler.fit_transform(X_train)
X_test[X_test.columns] = scaler.transform(X_test)

print(f"✅ Cleaned train shape: {X_train.shape}, test shape: {X_test.shape}")


NameError: name 'train_X_path' is not defined